# Calling External JAX Functions in JaxPlan.

This preliminary notebook discusses how to call external jax compiled function calls from within RDDL domain description files.

In [1]:
%pip install --quiet --upgrade pip
%pip install --quiet pyRDDLGym rddlrepository pyRDDLGym-jax

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Import the required packages:

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import jax
from flax import linen as nn

import pyRDDLGym
from pyRDDLGym.core.policy import RandomAgent
from pyRDDLGym_jax.core.simulator import JaxRDDLSimulator
from rddlrepository.core.manager import RDDLRepoManager

Let us now define the RDDL domain and instance description files containing the code with a function call. We will define a convolution layer with weights ``W`` and biases ``b`` to process the state variable: 

In [2]:
domain_text = """
domain my_domain {

    types {
        obj : object;
        kernel : object;
        feature : {@f1};
    };

    pvariables {
        state(obj, obj) : { state-fluent, real, default = 0.0 };
        action(obj, obj) : { action-fluent, real, default = 0.0 };

        W(kernel, kernel, feature, feature) : { non-fluent, real, default = 0.2 };
        b(feature) : { non-fluent, real, default = 0.0 };
    };
    
    cpfs {
        state'(?o, ?o2) = $CNNFunction[?o, ?o2](state(_, _), W(_, _, _, _), b(_)) + action(?o, ?o2);
    };
    
    reward = -(sum_{?o : obj, ?o2 : obj} pow[state'(?o, ?o2), 2]);
    
    action-preconditions {
        forall_{?o : obj, ?o2 : obj} [action(?o, ?o2) >= 0 ^ action(?o, ?o2) <= 1];
    };
}
"""

instance_text = """
non-fluents nf_simple {
    domain = my_domain;
    objects {
        obj : { o1, o2, o3 };
        kernel : { k1, k2 };
    };
}

instance simple_inst {
    domain = my_domain;
    non-fluents = nf_simple;

    init-state {
        state(o2, o2) = 1.0;
    };

    max-nondef-actions = 1;
    horizon = 5;
    discount = 1.0;
}
"""

# register the domain and instance with rddlrepository
manager = RDDLRepoManager(rebuild=True)
manager.register_domain("ExternalFuncDomainCNN", "standalone", domain_text, desc="domain with CNN", viz=None)
problem_info = manager.get_problem("ExternalFuncDomainCNN_standalone")
problem_info.register_instance("1", instance_text)
RDDLRepoManager(rebuild=True)

Domain <ExternalFuncDomainCNN> was successfully registered in rddlrepository with context <standalone>.
Instance <1> was successfully registered in rddlrepository for domain <ExternalFuncDomainCNN_standalone>.


The line of the code `state'(?o) = $CNNFunction[?o](state(_), ...);` calls an external JAX compiled function that applies a convolutional neural network to the current state.

Next, we must define the external function, which in this case defines the CNN in jax and flax; note the function must be JIT compilable in order to use JaxPlan:

In [3]:
class CNN(nn.Module):
    @nn.compact
    def __call__(self, x):
        return nn.Conv(
            features=1,
            kernel_size=(2, 2),
            padding="SAME",
            name="conv"
        )(x)

cnn = CNN()

@jax.jit
def cnn_function(state_vec, W, b):
    x = state_vec.reshape((1, 3, 3, 1))
    params = { 'params': { 'conv': { 'kernel': W, 'bias': b } } }
    value = cnn.apply(params, x).squeeze()
    return value

Finally, we need to instruct the JAX compiler to use the above function when compiling and executing the domain and instance; it will be included automatically as part of the computation graph during compilation:

In [4]:
env = pyRDDLGym.make("ExternalFuncDomainCNN_standalone", "1", 
                     backend=JaxRDDLSimulator,
                     backend_kwargs={'python_functions': {'CNNFunction': cnn_function}})

Let's execute the environment with a random policy:

In [5]:
agent = RandomAgent(action_space=env.action_space, num_actions=env.max_allowed_actions)
agent.evaluate(env, episodes=1, verbose=True)

initial state = 
     state___o1__o1 = 0.0  state___o1__o2 = 0.0  state___o1__o3 = 0.0 
     state___o2__o1 = 0.0  state___o2__o2 = 1.0  state___o2__o3 = 0.0 
     state___o3__o1 = 0.0  state___o3__o2 = 0.0  state___o3__o3 = 0.0 
    
--------------------------------------------------------------------------------
step   = 0
action = 
     action___o3__o3 = 0.934542179107666 
state  = 
     state___o1__o1 = 0.2        state___o1__o2 = 0.2       
     state___o1__o3 = 0.0        state___o2__o1 = 0.2       
     state___o2__o2 = 0.2        state___o2__o3 = 0.0       
     state___o3__o1 = 0.0        state___o3__o2 = 0.0       
     state___o3__o3 = 0.9345422 
reward = -1.0333690643310547
done   = False
--------------------------------------------------------------------------------
step   = 1
action = 
     action___o2__o2 = 0.916875958442688 
state  = 
     state___o1__o1 = 0.16000001   state___o1__o2 = 0.080000006 
     state___o1__o3 = 0.0          state___o2__o1 = 0.080000006 
     s

{'mean': np.float64(-4.201691746711731),
 'median': np.float64(-4.201691746711731),
 'min': np.float64(-4.201691746711731),
 'max': np.float64(-4.201691746711731),
 'std': np.float64(0.0)}